# Module 3: Intent Classifier

**Goal:** Classify user intent using a streamlined architecture (Crisis Detection -> Regex -> LLM)  
**Intents:** greeting, goodbye, gratitude, asking_mental_health_question, out_of_scope 

## Import Libraries

In [1]:
import os, re, json, random
import numpy as np
import pandas as pd
from dotenv import load_dotenv
from groq import Groq

In [2]:
load_dotenv()
GROQ_API_KEY = os.getenv('GROQ_API_KEY')
if not GROQ_API_KEY:
    raise ValueError('GROQ_API_KEY not found.')

GROQ_MODEL = 'llama-3.3-70b-versatile'
LLM_CONFIDENCE_THRESHOLD = 0.50
SEED = 42
INTENTS = ['greeting', 'goodbye', 'gratitude', 'asking_mental_health_question', 'out_of_scope']

client = Groq(api_key=GROQ_API_KEY)
print(f'✅ Groq client initialized | Model: {GROQ_MODEL}')

✅ Groq client initialized | Model: llama-3.3-70b-versatile


## Regex / Keyword Matcher

In [7]:
KEYWORD_PATTERNS = {
    'greeting': [r'\bhi\b', r'\bhello\b', r'\bhey\b', r'\bgood morning\b',
                 r'\bgood evening\b', r'\bgood afternoon\b', r'\bhow are you\b',
                 r'\bhowdy\b', r'\bgreetings\b'],
    'goodbye':  [r'\bbye\b', r'\bgoodbye\b', r'\bfarewell\b', r'\bsee you\b',
                 r'\btake care\b', r'\bgoodnight\b', r'\bi have to go\b',
                 r'\bi need to go\b', r'\btalk to you later\b', r'\bcatch you later\b'],
    'gratitude': [r'\bthank you\b', r'\bthanks\b', r'\bthank u\b',
                  r'\bi appreciate\b', r'\bgrateful\b', r'\bmany thanks\b']
}

def regex_match(text):
    text_lower = text.lower().strip()
    if len(text_lower.split()) > 4:
        return None
    for intent, patterns in KEYWORD_PATTERNS.items():
        for pattern in patterns:
            if re.search(pattern, text_lower):
                return intent
    return None

tests = ['Hi', 'Goodbye friend', 'Thank you so much', 'I feel anxious', 'What is the weather?']
print('Regex test:')
for t in tests:
    print(f'  "{t}" → {regex_match(t) or "no match"}')

Regex test:
  "Hi" → greeting
  "Goodbye friend" → goodbye
  "Thank you so much" → gratitude
  "I feel anxious" → no match
  "What is the weather?" → no match


## Crisis Detection

In [8]:
CRISIS_KEYWORDS = [
    'kill myself', 'end my life', 'want to die', 'suicide', 'suicidal',
    'hurt myself', 'self harm', 'self-harm', 'cutting myself',
    'want to hurt myself', 'no reason to live', 'better off dead',
    'overdose', 'jump off', 'hang myself',
    'أريد الموت', 'انتحار', 'أؤذي نفسي', 'أقتل نفسي', 'لا أريد العيش',
    'me suicider', 'me tuer', 'mettre fin à ma vie',
    'mich umbringen', 'suizid', 'selbstmord',
    'suicidarme', 'matarme', 'quiero morir',
]

CRISIS_RESPONSE = (
    "I'm really concerned about what you've shared, and I want you to know you're not alone. "
    "Please reach out to a crisis helpline immediately — they are available 24/7:\n\n"
    "🆘 International Association for Suicide Prevention: https://www.iasp.info/resources/Crisis_Centres/\n"
    "🆘 Crisis Text Line: Text HOME to 741741 (US)\n"
    "🆘 Befrienders Worldwide: https://www.befrienders.org\n\n"
    "Please talk to someone right now. You matter."
)

def check_crisis(text):
    text_lower = text.lower()
    return any(kw in text_lower for kw in CRISIS_KEYWORDS)

tests = [('I want to kill myself', True), ('I feel anxious', False), ('أريد الموت', True)]
print('Crisis detection test:')
for text, expected in tests:
    result = check_crisis(text)
    status = '✅' if result == expected else '❌'
    print(f'  {status} crisis={result} | "{text}"')


Crisis detection test:
  ✅ crisis=True | "I want to kill myself"
  ✅ crisis=False | "I feel anxious"
  ✅ crisis=True | "أريد الموت"


## Combined LLM Call

In [9]:
COMBINED_SYSTEM_PROMPT = """
You are an assistant for a multilingual mental health support chatbot.
Analyze the user message and return a JSON object with exactly these 4 fields:

1. "intent": Classify into exactly one of:
   - greeting
   - goodbye
   - gratitude
   - asking_mental_health_question
   - out_of_scope

2. "translated": The message in clean English.
   - If non-English: translate it. Preserve exact emotional tone and urgency. NEVER soften distress.
   - If already English: return as-is but fix typos and grammar. Never change the meaning.

3. "language": The TRUE language ISO code (en, ar, fr, de, es, it, pt, ru, zh, ja, tr, hi, nl, pl, bg, el, sw, vi, ur, th).
   - You receive a hint from our detector — treat it as suggestion only.
   - Use your own judgment, especially for short text where detectors fail.
   - If ambiguous (e.g. single word like "hi"), default to "en".

4. "confidence": Float 0-1 for intent certainty.

Return ONLY valid JSON:
{"intent": "...", "translated": "...", "language": "...", "confidence": 0.0}

Examples:
Message: "Hi" | hint: "zh" -> {"intent": "greeting", "translated": "Hi", "language": "en", "confidence": 0.99}
Message: "أشعر بالقلق الشديد ولا أستطيع النوم" | hint: "ar" -> {"intent": "asking_mental_health_question", "translated": "I feel very anxious and cannot sleep", "language": "ar", "confidence": 0.98}
Message: "Je me sens très triste et seul" | hint: "sw" -> {"intent": "asking_mental_health_question", "translated": "I feel very sad and lonely", "language": "fr", "confidence": 0.97}
Message: "i cant stpo crying and i dont knwo wat to do" | hint: "en" -> {"intent": "asking_mental_health_question", "translated": "I can't stop crying and I don't know what to do", "language": "en", "confidence": 0.98}
Message: "مع السلامة" | hint: "ar" -> {"intent": "goodbye", "translated": "Goodbye", "language": "ar", "confidence": 0.99}
Message: "What is the weather today?" | hint: "en" -> {"intent": "out_of_scope", "translated": "What is the weather today?", "language": "en", "confidence": 0.99}
Message: "I said goodbye to my old bad habits" | hint: "en" -> {"intent": "asking_mental_health_question", "translated": "I said goodbye to my old bad habits", "language": "en", "confidence": 0.95}
"""

def analyze_message(text, language_hint, max_retries=3):
    """Layer 2: Single Groq call — 4 jobs in one."""
    for attempt in range(max_retries):
        try:
            response = client.chat.completions.create(
                model=GROQ_MODEL,
                messages=[
                    {'role': 'system', 'content': COMBINED_SYSTEM_PROMPT},
                    {'role': 'user',   'content': f'Message: "{text}" | hint: "{language_hint}"'}
                ],
                temperature=0.0, max_tokens=200
            )
            raw    = response.choices[0].message.content.strip()
            raw    = re.sub(r'```json|```', '', raw).strip()
            parsed = json.loads(raw)
            if parsed.get('intent') not in INTENTS:
                raise ValueError(f"Invalid intent: {parsed.get('intent')}")
            return {
                'intent':     parsed['intent'],
                'translated': parsed.get('translated', text),
                'language':   parsed.get('language', language_hint),
                'confidence': float(parsed.get('confidence', 0.5)),
                'layer_used': 'Layer 2 (LLM)'
            }
        except Exception as e:
            print(f'  Attempt {attempt+1} failed: {e}')
            if attempt == max_retries - 1:
                return {'intent': 'out_of_scope', 'translated': text,
                        'language': language_hint, 'confidence': 0.0,
                        'layer_used': 'Layer 2 (LLM — fallback)'}

# Quick test
tests = [
    ('أشعر بالقلق الشديد', 'ar'),
    ('i cant stpo crying',  'en'),
    ('Hi',                  'zh'),
]
print('Layer 2 test:')
for text, hint in tests:
    r = analyze_message(text, hint)
    print(f'  [{r["language"]}] {r["intent"]} | "{r["translated"]}" | {r["confidence"]:.0%}')

Layer 2 test:
  [ar] asking_mental_health_question | "I feel very anxious" | 98%
  [en] asking_mental_health_question | "I can't stop crying" | 98%
  [en] greeting | "Hi" | 99%


## Full classify_intent()

In [10]:
def classify_intent(text, language_hint='en'):
    """
    Unified local/API classifier wrapper.
    All paths return a consistent dictionary structure.
    """
    regex_result = regex_match(text)
    if regex_result:
        return {'intent': regex_result, 'translated': text, 'language': language_hint,
                'confidence': 1.0, 'layer_used': 'Layer 1 (Regex)'}

    return analyze_message(text, language_hint)

print('classify_intent() defined ✅')

classify_intent() defined ✅


## Language-Aware Router

In [11]:
DIRECT_RESPONSES = {
    'greeting': {
        'en': ["Hello! I'm here to support you. How are you feeling today?",
               "Hi there! I'm your mental health support assistant. What's on your mind?"],
        'ar': ["مرحباً! أنا هنا لدعمك. كيف تشعر اليوم؟"],
        'fr': ["Bonjour! Je suis là pour vous soutenir. Comment vous sentez-vous aujourd'hui?"],
        'de': ["Hallo! Ich bin hier um Sie zu unterstützen. Wie fühlen Sie sich heute?"],
        'es': ["¡Hola! Estoy aquí para apoyarte. ¿Cómo te sientes hoy?"],
        'it': ["Ciao! Sono qui per supportarti. Come ti senti oggi?"],
        'pt': ["Olá! Estoy aqui para te apoiar. Como você está se sentindo hoje?"],
        'ru': ["Привет! Я здесь, чтобы поддержать вас."],
        'zh': ["你好！我在这里支持你。你今天感觉怎么样？"],
        'ja': ["こんにちは！あなたをサポートするためにここにいます。"],
        'default': ["Hello! I'm here to support you. How are you feeling today?"]
    },
    'goodbye': {
        'en': ["Take care of yourself. Support is always here when you need it.",
               "Goodbye! I hope our conversation was helpful. Be well."],
        'ar': ["اعتنِ بنفسك. الدعم دائماً هنا عندما تحتاجه."],
        'fr': ["Prenez soin de vous. Le soutien est toujours là."],
        'de': ["Pass auf dich auf. Unterstützung ist immer da."],
        'es': ["Cuídate. El apoyo siempre está aquí."],
        'it': ["Prenditi cura di te. Il supporto é sempre qui."],
        'pt': ["Cuide-se. O apoio está sempre aqui."],
        'ru': ["Берегите себя. Поддержка всегда здесь."],
        'zh': ["保重。支持随时都在。"],
        'ja': ["お体に気をつけて。いつでもここでサポートします。"],
        'default': ["Take care of yourself. Support is always here when you need it."]
    },
    'gratitude': {
        'en': ["You're very welcome. I'm always here for you.",
               "I'm glad I could help. Please reach out anytime."],
        'ar': ["على الرحب والسعة. أنا دائماً هنا من أجلك."],
        'fr': ["De rien. Je suis toujours là pour vous."],
        'de': ["Gern geschehen. Ich bin immer für Sie da."],
        'es': ["De nada. Siempre estoy aquí para ti."],
        'it': ["Prego. Sono sempre qui per te."],
        'pt': ["De nada. Estou sempre aqui para você."],
        'ru': ["Пожалуйста. Я всегда здесь для вас."],
        'zh': ["不客气。我随时都在这里为你服务。"],
        'ja': ["どういたしまして。いつでもここにいます。"],
        'default': ["You're very welcome. I'm always here for you."]
    }
}

OUT_OF_SCOPE_RESPONSES = {
    'en': "I'm specialized in mental health support. Is there anything related to your emotional wellbeing I can help with?",
    'ar': "أنا متخصص في دعم الصحة النفسية. هل هناك شيء يتعلق بصحتك النفسية يمكنني مساعدتك به؟",
    'fr': "Je suis spécialisé dans le soutien en santé mentale. Y a-t-il quelque chose lié à votre bien-être émotionnel?",
    'de': "Ich bin auf psychische Gesundheitsunterstützung spezialisiert. Kann ich bei Ihrem Wohlbefinden helfen?",
    'es': "Me especializo en apoyo de salud mental. ¿Hay algo relacionado con tu bienestar emocional?",
    'default': "I'm specialized in mental health support. Is there anything related to your emotional wellbeing I can help with?"
}

def get_direct_response(intent, language):
    options = DIRECT_RESPONSES[intent].get(language, DIRECT_RESPONSES[intent]['default'])
    return random.choice(options)

def get_out_of_scope_response(language):
    return OUT_OF_SCOPE_RESPONSES.get(language, OUT_OF_SCOPE_RESPONSES['default'])

def router(user_text, language_hint='en'):
    """
    Full pipeline: crisis → classify → route.
    Returns: {route, intent, translated, language, confidence, layer_used, response}
    """
    # Step 0: Crisis Check
    if check_crisis(user_text):
        return {'route': 'crisis', 'intent': 'crisis', 'translated': user_text,
                'language': language_hint, 'confidence': 1.0,
                'layer_used': 'Step 0 (Crisis)', 'response': CRISIS_RESPONSE}

    c = classify_intent(user_text, language_hint)
    intent, translated = c['intent'], c['translated']
    language, confidence, layer_used = c['language'], c['confidence'], c['layer_used']

    if confidence < LLM_CONFIDENCE_THRESHOLD:
        return {'route': 'clarification', 'intent': intent, 'translated': translated,
                'language': language, 'confidence': confidence, 'layer_used': layer_used,
                'response': "I want to make sure I understand you correctly. Could you rephrase that?"}

    if intent in ('greeting', 'goodbye', 'gratitude'):
        return {'route': 'direct', 'intent': intent, 'translated': translated,
                'language': language, 'confidence': confidence, 'layer_used': layer_used,
                'response': get_direct_response(intent, language)}

    elif intent == 'asking_mental_health_question':
        return {'route': 'rag_pipeline', 'intent': intent,
                'translated': translated,  # clean English → emotion + RAG
                'language': language,      # verified language
                'confidence': confidence, 'layer_used': layer_used,
                'response': 'RAG_PIPELINE'}

    else:
        return {'route': 'out_of_scope', 'intent': intent, 'translated': translated,
                'language': language, 'confidence': confidence, 'layer_used': layer_used,
                'response': get_out_of_scope_response(language)}

print('router() defined ✅')

router() defined ✅


## Full System Test

In [12]:
test_cases = [
    ('Hi there!',                                  'en', 'greeting'),
    ('Good morning, how are you?',                 'en', 'greeting'),
    ('Goodbye, talk to you later',                 'en', 'goodbye'),
    ('Thank you so much for your help',            'en', 'gratitude'),
    ("I've been feeling really anxious lately",    'en', 'asking_mental_health_question'),
    ('How do I cope with depression?',             'en', 'asking_mental_health_question'),
    ('I feel like giving up on everything',        'en', 'asking_mental_health_question'),
    ('i cant stpo crying wat do i do',             'en', 'asking_mental_health_question'),
    ('I want to kill myself',                      'en', 'crisis'),
    ('What is the weather today?',                 'en', 'out_of_scope'),
    ('Recommend me a good movie',                  'en', 'out_of_scope'),
    ('أشعر بالقلق الشديد هذه الأيام',              'ar', 'asking_mental_health_question'),
    ('مرحبا',                                      'ar', 'greeting'),
    ('Je me sens très triste et seul',             'fr', 'asking_mental_health_question'),
    ('Hi',                                         'zh', 'greeting'),
    ('I said goodbye to my old bad habits',        'en', 'asking_mental_health_question'),
]

print(f"{'Input':<45} {'Expected':<32} {'Got':<32} {'Lang':>4} {'Layer':<22} {'Conf':>6}  Status")
print('-' * 160)

correct = 0
for text, lang_hint, expected in test_cases:
    result = router(text, language_hint=lang_hint)
    got    = result['intent']
    status = '✅' if got == expected else '❌'
    if got == expected: correct += 1
    print(f"{text:<45} {expected:<32} {got:<32} {result['language']:>4} {result['layer_used']:<22} {result['confidence']:>6.2%}  {status}")

print('-' * 160)
print(f'\nAccuracy: {correct}/{len(test_cases)} ({correct/len(test_cases):.0%})')


Input                                         Expected                         Got                              Lang Layer                    Conf  Status
----------------------------------------------------------------------------------------------------------------------------------------------------------------
Hi there!                                     greeting                         greeting                           en Layer 1 (Regex)        100.00%  ✅
Good morning, how are you?                    greeting                         greeting                           en Layer 2 (LLM)          99.00%  ✅
Goodbye, talk to you later                    goodbye                          goodbye                            en Layer 2 (LLM)          99.00%  ✅
Thank you so much for your help               gratitude                        gratitude                          en Layer 2 (LLM)          99.00%  ✅
I've been feeling really anxious lately       asking_mental_health_question    aski

## Summary

| Item | Detail |
|---|---|
| **Step 0** | Crisis detection — local string searching, zero API cost |
| **Layer 1** | Regex — instant, handling short English expressions locally |
| **Layer 2** | Combined LLM — unified execution for intent + translation + language verification |
| **Layer 3** | Language-aware router — maps responses efficiently based on final routing flags |
| **Public API** | `router(user_text, language_hint) → dict` |

### What flows to Module 4:
```python
result = router(user_text, language_hint)
# result['translated'] → clean English → emotion detection + RAG
# result['language']   → verified language → LLM responds in this language
# result['response'] == 'RAG_PIPELINE' → signal to call Module 4
```